In [2]:
import duckdb
import pandas as pd
from lifetimes.utils import summary_data_from_transaction_data

con = duckdb.connect('../../warehouse.duckdb')

orders = con.execute("""
    select
        user_id,
        order_date,
        revenue
    from main.fct_orders
""").fetchdf()

summary = summary_data_from_transaction_data(
    orders,
    customer_id_col='user_id',
    datetime_col='order_date',
    monetary_value_col='revenue',
    observation_period_end=orders['order_date'].max()
)

summary.head()

,frequency,recency,T,monetary_value
user_id,,,,
u_000001,4.0,379.0,681.0,110.785
u_000010,0.0,0.0,505.0,0.000
u_000015,4.0,266.0,438.0,79.120
u_000024,0.0,0.0,417.0,0.000
u_000027,5.0,172.0,670.0,67.376


In [3]:
from lifetimes import BetaGeoFitter, GammaGammaFitter

# BG/NBD needs customers with frequency >= 0, fits on everyone
bgf = BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(summary['frequency'], summary['recency'], summary['T'])

print(bgf)

# Gamma-Gamma needs repeat customers only (frequency > 0) to model spend
returning_customers = summary[summary['frequency'] > 0]

ggf = GammaGammaFitter(penalizer_coef=0.001)
ggf.fit(returning_customers['frequency'], returning_customers['monetary_value'])

print(ggf)

<lifetimes.BetaGeoFitter: fitted with 1650 subjects, a: 1.85, alpha: 19.63, b: 2.87, r: 0.30>
<lifetimes.GammaGammaFitter: fitted with 861 subjects, p: 12.10, q: 2.61, v: 11.51>
